# 🚀 Colab / Kaggle - Auto-Annotation Setup

Sử dụng GPU miễn phí trên Cloud để tăng tốc độ dán nhãn tự động (Grounding DINO).

### 1. Bật GPU
- **Colab**: Menu `Runtime` -> `Change runtime type` -> `T4 GPU`.
- **Kaggle**: Panel bên phải `Settings` -> `Accelerator` -> `GPU T4 x2`.

### 2. Chuẩn bị
- Sau khi nén `src` thành `src.zip` và `output/frames_clear` thành `frames.zip`, hãy tải chúng lên vùng lưu trữ của Colab/Kaggle.

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install autodistill autodistill-grounding-dino supervision roboflow -q
!pip install "transformers<4.40.0" "tokenizers==0.15.2" -q

# 2. TỰ ĐỘNG KIỂM TRA VÀ RESTART SESSION (Bắt buộc để fix lỗi BertModel)
import transformers
from packaging import version
if version.parse(transformers.__version__) >= version.parse("4.40.0"):
    print(f"\u274c Transformers version {transformers.__version__} là bản mới, cần hạ cấp và restart!")
    print("\ud83d? Đang tự động Restart Session... Sau khi Restart, bạn hãy chạy lại cell này một lần nữa né.")
    import os
    os.kill(os.getpid(), 9)
else:
    print(f"\u2705 Transformers version {transformers.__version__} đã sẵn sàng và t💥ng thích!")

# 3. Giải nén dữ liệu
!unzip -q src.zip -d .
!unzip -q frames.zip -d ./output/frames_clear

print("\u2705 Setup và Giải nén xong! Sẵn sàng chạy auto-annotation.")

# 02b - Auto-Annotation with Grounding DINO
## Bradford Bulls - AI Sponsorship Exposure Valuation System

**Problem:** 6000+ frames is too many to annotate manually.

**Solution:** 
1. Select ~500-800 most diverse frames (skip near-duplicates)
2. Use **Grounding DINO** (zero-shot object detection) to auto-detect logos by text prompt
3. Generate YOLO-format annotation files
4. Upload images + annotations to Roboflow → only need to **review & correct**

This reduces manual work from drawing 6000+ bounding boxes to just reviewing/fixing pre-drawn ones.

## 1. Install Dependencies

In [1]:
# Cài đặt LOCAL (Đã có bản trên Colab ở phần Setup phía trên)
# !pip install autodistill autodistill-grounding-dino supervision -q

In [2]:
import sys
sys.path.insert(0, "..")

import cv2
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from tqdm import tqdm

from src.config import (
    FRAMES_CLEAR_DIR, METADATA_DIR, OUTPUT_DIR,
    SPONSOR_LABELS, DEVICE
)

print(f"Device: {DEVICE}")
print(f"Frames available: {len(list(FRAMES_CLEAR_DIR.glob('frame_*.jpg')))}")

## 2. Select Diverse Frame Subset

From 6000+ frames, select ~500-800 most diverse frames using visual similarity filtering.
This avoids annotating near-identical frames and gives better training data diversity.

In [3]:
import imagehash
from PIL import Image

# ============================================================
# CONFIG: How many frames to select for annotation
# ============================================================
TARGET_ANNOTATION_COUNT = 600  # Adjust as needed (500-800 recommended)
DIVERSITY_HASH_THRESHOLD = 8  # Higher = more similar frames allowed

all_frames = sorted(FRAMES_CLEAR_DIR.glob("frame_*.jpg"))
print(f"Total clear frames: {len(all_frames)}")

# Strategy: Walk through frames, keep only those that are visually different
# from the last kept frame (using perceptual hash)
selected_frames = []
last_hash = None

for frame_path in tqdm(all_frames, desc="Selecting diverse frames"):
    img = Image.open(frame_path)
    current_hash = imagehash.phash(img)
    
    if last_hash is None or (current_hash - last_hash) >= DIVERSITY_HASH_THRESHOLD:
        selected_frames.append(frame_path)
        last_hash = current_hash

print(f"\nAfter diversity filtering: {len(selected_frames)} frames")

# If still too many, subsample evenly
if len(selected_frames) > TARGET_ANNOTATION_COUNT:
    step = len(selected_frames) / TARGET_ANNOTATION_COUNT
    indices = [int(i * step) for i in range(TARGET_ANNOTATION_COUNT)]
    selected_frames = [selected_frames[i] for i in indices]
    print(f"After subsampling: {len(selected_frames)} frames")

print(f"\nSelected {len(selected_frames)} frames for auto-annotation")

In [4]:
# Copy selected frames to a separate directory for annotation
ANNOTATION_DIR = OUTPUT_DIR / "frames_for_annotation"
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

for fp in tqdm(selected_frames, desc="Copying selected frames"):
    shutil.copy2(fp, ANNOTATION_DIR / fp.name)

print(f"Copied {len(selected_frames)} frames to: {ANNOTATION_DIR}")

## 3. Auto-Annotate with Grounding DINO

Grounding DINO is a **zero-shot** object detector — it can find objects by text description without any training.

We use text prompts like `"AON logo"`, `"sponsor logo on jersey"`, `"text on shirt"` to detect logo regions automatically.

In [5]:
from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology

# ============================================================
# ONTOLOGY: Map text prompts → label classes
# ============================================================
ontology = CaptionOntology({
    "AON logo": "aon",
    "AON text on jersey": "aon",
    "MCP logo": "mcp",
    "MCP text on jersey": "mcp",
    "Cedar Court Hotels logo": "cch_cedar_court",
    "ChadLaw logo": "chadlaw",
    "ATM Hospitality logo": "atm_hospitality",
    "EM workwear logo": "em_workwear",
    "Fairway Flooring logo": "fairway_flooring",
    "KLG logo": "klg",
    "MNA logo": "mna_cladding",
    "MNA text on shorts": "mna_support",
    "Bartercard logo": "bartercard",
    "Top Notch logo": "top_notch",
    "Romantica logo": "romantica_beds",
    "sponsor logo on rugby jersey": "sponsor_logo",
    "text on rugby shirt": "sponsor_logo",
    "brand logo on sportswear": "sponsor_logo",
})

print("Ontology configured with", len(ontology.promptMap), "prompts")
print("\nPrompt → Label mappings:")
for prompt, label in ontology.promptMap:
    print(f"  '{prompt}' → {label}")

In [6]:
# ============================================================
# RUN AUTO-ANNOTATION
# ============================================================
AUTOLABEL_OUTPUT = OUTPUT_DIR / "auto_annotated"
AUTOLABEL_OUTPUT.mkdir(parents=True, exist_ok=True)

base_model = GroundingDINO(
    ontology=ontology,
    box_threshold=0.2,
    text_threshold=0.15,
)

base_model.label(
    input_folder=str(ANNOTATION_DIR),
    output_folder=str(AUTOLABEL_OUTPUT),
)

print(f"\nAuto-annotation complete!")

## 4. Preview Auto-Annotations

In [ ]:
class_names = list(dict.fromkeys([label for prompt, label in ontology.promptMap]))
print(f"Class names: {class_names}\n")

def visualize_annotation(image_path, label_path, class_names):
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    ax.imshow(img_rgb)
    if label_path.exists():
        with open(label_path) as f: lines = f.readlines()
        colors = plt.cm.Set3(np.linspace(0, 1, len(class_names)))
        for line in lines:
            parts = line.strip().split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = [float(x) for x in parts[1:5]]
            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            label = class_names[cls_id] if cls_id < len(class_names) else f"class_{cls_id}"
            color = colors[cls_id % len(colors)]
            rect = patches.Rectangle((x1, y1), bw*w, bh*h, linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, label, color=color, fontweight='bold', backgroundcolor='black')
    ax.axis("off")
    plt.show()